# PAI最終課題：2軸アームの到達動作（MuJoCo + PPO）
ランタイムはCPUでOK。上から順に実行してください。

In [ ]:
!pip install -q mujoco gymnasium stable-baselines3 imageio imageio-ffmpeg
!pip uninstall -y -q gym

In [ ]:
%%writefile arm2_env.py
"""
2自由度（2軸）平面アームの強化学習環境
- 物理シミュレーション : MuJoCo
- 環境インターフェース : Gymnasium
- 可視化              : matplotlib（MuJoCoのOpenGL描画を使わないため、CPU環境でも動く）
タスク: ランダムな位置に置かれた目標（緑の球）に、アームの手先を到達させる
"""
import numpy as np
import mujoco
import gymnasium as gym
from gymnasium import spaces
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---- ロボットの設計図（MJCF） 土台→関節1→リンク1(0.2m)→関節2→リンク2(0.2m)→手先 ----
XML = """
<mujoco model="arm2dof">
  <option timestep="0.01" gravity="0 0 0"/>
  <worldbody>
    <geom type="plane" size="1 1 0.1"/>
    <body name="link1" pos="0 0 0.05">
      <joint name="j1" type="hinge" axis="0 0 1" damping="0.5"/>
      <geom type="capsule" fromto="0 0 0 0.2 0 0" size="0.02"/>
      <body name="link2" pos="0.2 0 0">
        <joint name="j2" type="hinge" axis="0 0 1" limited="true" range="-150 150" damping="0.5"/>
        <geom type="capsule" fromto="0 0 0 0.2 0 0" size="0.018"/>
        <site name="tip" pos="0.2 0 0" size="0.02"/>
      </body>
    </body>
    <body name="target" mocap="true" pos="0.2 0.2 0.05">
      <geom type="sphere" size="0.025" contype="0" conaffinity="0"/>
    </body>
  </worldbody>
  <actuator>
    <motor joint="j1" ctrllimited="true" ctrlrange="-1 1"/>
    <motor joint="j2" ctrllimited="true" ctrlrange="-1 1"/>
  </actuator>
</mujoco>
"""

EPISODE_STEPS = 200      # 1エピソード = 200ステップ = 4秒
SUCCESS_DIST = 0.02      # 手先と目標の距離が2cm未満なら成功


class Arm2Env(gym.Env):
    metadata = {"render_modes": ["rgb_array"], "render_fps": 50}

    def __init__(self, render_mode=None):
        self.model = mujoco.MjModel.from_xml_string(XML)
        self.data = mujoco.MjData(self.model)
        # 行動: 2つのモータへのトルク指令（-1〜+1）
        self.action_space = spaces.Box(-1, 1, (2,), np.float32)
        # 観測: cos(q)2, sin(q)2, 関節速度2, 目標位置2, 手先-目標のズレ2 = 10次元
        self.observation_space = spaces.Box(-np.inf, np.inf, (10,), np.float64)
        self.render_mode = render_mode
        self.fig = None
        self.t = 0

    def _obs(self):
        q = self.data.qpos
        tip = self.data.site("tip").xpos[:2]
        tgt = self.data.mocap_pos[0][:2]
        return np.concatenate([np.cos(q), np.sin(q), self.data.qvel, tgt, tip - tgt])

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        mujoco.mj_resetData(self.model, self.data)
        self.data.qpos[:] = self.np_random.uniform(-0.1, 0.1, 2)   # 初期姿勢を少しランダムに
        r = self.np_random.uniform(0.1, 0.38)                      # 目標は必ず届く範囲に置く
        th = self.np_random.uniform(-np.pi, np.pi)
        self.data.mocap_pos[0] = [r * np.cos(th), r * np.sin(th), 0.05]
        mujoco.mj_forward(self.model, self.data)
        self.t = 0
        return self._obs(), {}

    def step(self, a):
        self.data.ctrl[:] = a
        for _ in range(2):                                          # 0.01s × 2 = 0.02s 進める
            mujoco.mj_step(self.model, self.data)
        self.t += 1
        obs = self._obs()
        dist = float(np.linalg.norm(obs[-2:]))
        # 報酬: 目標との距離を減点 ＋ 大きなトルクに小さなペナルティ（滑らかな動作のため）
        reward = -dist - 0.01 * float(np.sum(np.square(a)))
        return obs, reward, False, self.t >= EPISODE_STEPS, {"dist": dist}

    def render(self):
        if self.fig is None:
            self.fig, self.ax = plt.subplots(figsize=(4.8, 4.8), dpi=100)
        ax = self.ax
        ax.clear()
        ax.set_xlim(-0.45, 0.45); ax.set_ylim(-0.45, 0.45)
        ax.set_aspect("equal"); ax.grid(alpha=0.3)
        elbow = self.data.body("link2").xpos[:2]
        tip = self.data.site("tip").xpos[:2]
        tgt = self.data.mocap_pos[0][:2]
        ax.plot([0, elbow[0]], [0, elbow[1]], lw=8, color=(.2, .4, .8), solid_capstyle="round")
        ax.plot([elbow[0], tip[0]], [elbow[1], tip[1]], lw=7, color=(.8, .4, .2), solid_capstyle="round")
        ax.plot(0, 0, "ko", ms=8)
        ax.add_patch(plt.Circle(tgt, 0.025, color="lime", alpha=0.6))
        ax.set_title(f"t = {self.t * 0.02:.2f} s   dist = {np.linalg.norm(tip - tgt):.3f} m")
        self.fig.canvas.draw()
        return np.asarray(self.fig.canvas.buffer_rgba())[:, :, :3].copy()

    def close(self):
        if self.fig is not None:
            plt.close(self.fig)


In [ ]:
%%writefile train.py
"""PPO（Stable-Baselines3, MLP方策）で2軸アームの到達動作を学習する"""
import argparse
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from arm2_env import Arm2Env

p = argparse.ArgumentParser()
p.add_argument("--steps", type=int, default=500_000)   # 総学習ステップ数
p.add_argument("--n_envs", type=int, default=8)        # 並列環境数
p.add_argument("--seed", type=int, default=0)
p.add_argument("--out", default="ppo_arm2")
args = p.parse_args()

# monitor_dir に各エピソードの報酬が記録される → evaluate.py で学習曲線を描く
env = make_vec_env(Arm2Env, n_envs=args.n_envs, seed=args.seed, monitor_dir="logs/monitor")
model = PPO("MlpPolicy", env, verbose=1, n_steps=1024, batch_size=256,
            seed=args.seed, device="cpu", tensorboard_log="logs/tb")
model.learn(total_timesteps=args.steps)
model.save(args.out)
print(f"saved: {args.out}.zip")


In [ ]:
%%writefile evaluate.py
"""学習済みモデルの評価・学習曲線・デモ動画の作成
出力: results/results.json, results/learning_curve.png, results/arm2_result.mp4
"""
import argparse, json, os
import numpy as np
import imageio
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.results_plotter import load_results, ts2xy
from arm2_env import Arm2Env, SUCCESS_DIST

p = argparse.ArgumentParser()
p.add_argument("--model", default="ppo_arm2")
p.add_argument("--episodes", type=int, default=100)      # 定量評価の試行数
p.add_argument("--video_episodes", type=int, default=20) # 20回 × 4秒 = 約80秒の動画
args = p.parse_args()
os.makedirs("results", exist_ok=True)

def run(policy, n, seed0=1000):
    """n回試行し、最終距離・成功・到達時間を返す（seedは学習時と別）"""
    env = Arm2Env(); finals, reach_t = [], []
    for ep in range(n):
        obs, _ = env.reset(seed=seed0 + ep); t_hit = None
        for t in range(200):
            obs, _, _, _, info = env.step(policy(obs, env))
            if t_hit is None and info["dist"] < SUCCESS_DIST:
                t_hit = (t + 1) * 0.02
        finals.append(info["dist"]); reach_t.append(t_hit)
    finals = np.array(finals)
    hits = [x for x in reach_t if x is not None]
    return {"success_rate": float(np.mean(finals < SUCCESS_DIST)),
            "final_dist_mean_m": float(finals.mean()),
            "final_dist_median_m": float(np.median(finals)),
            "reach_time_mean_s": float(np.mean(hits)) if hits else None}

model = PPO.load(args.model, device="cpu")
ppo = run(lambda o, e: model.predict(o, deterministic=True)[0], args.episodes)
rnd = run(lambda o, e: e.action_space.sample(), args.episodes)        # 比較用：ランダム行動
res = {"episodes": args.episodes, "success_threshold_m": SUCCESS_DIST, "ppo": ppo, "random": rnd}
json.dump(res, open("results/results.json", "w"), indent=2, ensure_ascii=False)
print(json.dumps(res, indent=2, ensure_ascii=False))

# 学習曲線（Monitorログがある場合）
if os.path.isdir("logs/monitor"):
    x, y = ts2xy(load_results("logs/monitor"), "timesteps")
    k = 50
    ys = np.convolve(y, np.ones(k) / k, mode="valid") if len(y) > k else y
    plt.figure(figsize=(6, 3.5))
    plt.plot(x[:len(ys)] if len(y) <= k else x[k - 1:], ys)
    plt.xlabel("timesteps"); plt.ylabel("episode reward (moving avg)")
    plt.title("Learning curve (PPO)"); plt.grid(alpha=.3); plt.tight_layout()
    plt.savefig("results/learning_curve.png", dpi=150)

# デモ動画
env = Arm2Env(render_mode="rgb_array"); frames = []
for ep in range(args.video_episodes):
    obs, _ = env.reset(seed=ep)
    for _ in range(200):
        obs, _, _, _, info = env.step(model.predict(obs, deterministic=True)[0])
        frames.append(env.render())
    print(f"ep{ep} final dist: {info['dist']:.3f} m")
imageio.mimsave("results/arm2_result.mp4", frames, fps=50)
print("saved: results/arm2_result.mp4")


In [ ]:
# 描画チェック
from arm2_env import Arm2Env
e = Arm2Env(render_mode='rgb_array'); e.reset(seed=0)
print('描画OK:', e.render().shape)

In [ ]:
# 学習（十数分〜）
!python train.py --steps 500000

In [ ]:
# 評価・学習曲線・動画（数分）
!python evaluate.py

In [ ]:
from IPython.display import Video, Image, display
display(Image('results/learning_curve.png'))
Video('results/arm2_result.mp4', embed=True, width=480)

In [ ]:
# ダウンロード（レポート用）
from google.colab import files
for f in ['results/arm2_result.mp4','results/learning_curve.png','results/results.json']:
    files.download(f)